# Apply RSS Feed Patch

Daily operator notebook. Run cells 1–4 to **review**, then cells 5–6 to **apply**.

**Prerequisites:** agent `.venv` active (`source .venv/bin/activate`)  
**API must be running:** `docker-compose up -d api` (port 8001)


In [49]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────
# Edit PATCH_FILE each morning; everything else is auto-detected from .env.

import os
from datetime import date
from pathlib import Path

# Path to today's additions YAML from the curator skill.
PATCH_FILE = Path(
    "~/Documents/Claude/Projects/Stumble Upon/rss-curator/"
    f"daily-reports/{date.today()}-additions.yaml"
).expanduser()

# Agent API — matches docker-compose port 8001 locally, or set AGENT_API_URL env var for cloud.
AGENT_API = os.environ.get("AGENT_API_URL", "https://agent.serendip.mountainman.tech")

# ── Load .env (best-effort) ──────────────────────────────────────────────────
_env_file = (
    Path(__file__).parent.parent.parent / ".env"
    if "__file__" in dir()
    else Path.cwd().parents[1] / ".env"
)
if _env_file.exists():
    for _line in _env_file.read_text().splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _, _v = _line.partition("=")
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

# ── Override here if not in .env ─────────────────────────────────────────────
# INTERNAL_API_TOKEN = "32f69a4bc906cd49624b9e9fef123fe126c407ce28d3f346fde151f85713ec92"
# DATABASE_URL       = "postgresql://stumble:ZQQ8r0cMeUExFfRMxEbU7RdIlcTC4Rb7@<vm-ip>:5432/stumble_ai"

INTERNAL_API_TOKEN = os.environ.get("INTERNAL_API_TOKEN", "")
DATABASE_URL = os.environ.get("DATABASE_URL", "")

print(f"Patch file : {PATCH_FILE}")
print(f"Exists     : {PATCH_FILE.exists()}")
print(f"Agent API  : {AGENT_API}")
print(f"Token set  : {'yes' if INTERNAL_API_TOKEN else 'NO -- uncomment override lines above'}")
print(f"DB URL set : {'yes' if DATABASE_URL else 'NO -- uncomment override lines above'}")

IndentationError: unindent does not match any outer indentation level (<string>, line 31)

In [45]:
# ── Cell 2: Load & parse the patch YAML ─────────────────────────────────────
try:
    import yaml
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml

if not PATCH_FILE.exists():
    raise FileNotFoundError(
        f"Patch file not found: {PATCH_FILE}\n"
        "Update PATCH_FILE in Cell 1 to point at today's curator output."
    )

patch = yaml.safe_load(PATCH_FILE.read_text())

additions = patch.get("additions") or []
removals = patch.get("removals") or []
watch = patch.get("watch") or []

print(f"Patch date  : {patch.get('date', 'unknown')}")
print(f"Additions   : {len(additions)}")
print(f"Removals    : {len(removals)}")
print(f"Watch       : {len(watch)}")

FileNotFoundError: Patch file not found: /Users/will/Documents/Claude/Projects/Stumble Upon/rss-curator/daily-reports/2026-04-19-additions.yaml
Update PATCH_FILE in Cell 1 to point at today's curator output.

In [46]:
# ── Cell 3: Review additions (read-only) ─────────────────────────────────────
RESET = "\033[0m"
BOLD = "\033[1m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
RED = "\033[31m"


def score_color(s):
    s = float(s) if s is not None else 0.0
    if s >= 0.80:
        return GREEN
    if s >= 0.65:
        return YELLOW
    return RED


print(f"{BOLD}{'URL':<52} {'Category':<14} Score  Justification{RESET}")
print("-" * 110)
for item in additions:
    url = item.get("url", "")
    cat = item.get("category", "general")
    score = item.get("editorial_score")
    just = item.get("justification", "")
    color = score_color(score)
    score_str = f"{score:.2f}" if score is not None else "n/a"
    print(f"{color}{url[:51]:<52}{RESET} {cat:<14} {color}{score_str:<6}{RESET} {just[:60]}")

if not additions:
    print("  (no additions)")

URL                                                  Category       Score  Justification
--------------------------------------------------------------------------------------------------------------
https://www.theatlantic.com/feed/channel/ideas/      culture        0.85   The Atlantic's Ideas section — political and cultural longfo
https://thequietus.com/feed                          music          0.86   Independent UK music and cultural-criticism magazine — disti
https://www.stereogum.com/feed/                      music          0.78   Long-running indie/alternative music site with original revi
https://anthropocenemagazine.org/feed/               nature         0.80   Anthropocene Magazine — research-driven environmental journa
https://www.core77.com/blog/rss.xml                  design         0.80   Core77 — long-running industrial design publication and comm
https://www.dezeen.com/feed/                         design         0.78   Dezeen — most influential architecture/interi

In [47]:
# ── Cell 4: Review removals & watch list (read-only) ─────────────────────────
if removals:
    print(f"{BOLD}REMOVALS -- will be marked dead in rss_feeds{RESET}")
    print("-" * 70)
    for item in removals:
        print(f"  {RED}x{RESET}  {item.get('url', '')}")
        print(f"      Reason: {item.get('reason', '')}")
else:
    print("No removals proposed.")
print()
if watch:
    print(f"{BOLD}WATCH LIST -- display only, no action{RESET}")
    print("-" * 70)
    for item in watch:
        print(f"  {YELLOW}!{RESET}  {item.get('url', '')}")
        print(f"      Concern: {item.get('concern', '')}")
else:
    print("No watch items.")

No removals proposed.

WATCH LIST -- display only, no action
----------------------------------------------------------------------
  !  https://blog.pragmaticengineer.com/rss/
      Concern: Cadence 2 items/30d (just over the ≥1/14d floor) — free-tier of a paid newsletter. Watch in case the public feed slows further.
  !  https://www.cntraveler.com/feed/rss
      Concern: LLM score 0.65 — exactly at the cutoff. Reassess after first ingestion run.
  !  https://bigthink.com/feed/
      Concern: LLM score 0.68 — borderline mainstream content, sometimes drifts toward listicle. Reassess after first ingestion run.


---
## Apply section

Review the tables above before running the cells below.


In [48]:
# ── Cell 5: Apply additions via POST /internal/feeds ────────────────────────
try:
    import httpx
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

if not INTERNAL_API_TOKEN:
    raise RuntimeError("INTERNAL_API_TOKEN is not set -- update Cell 1.")

total_queued = total_skipped = 0
all_errors = []

if not additions:
    print("Nothing to add.")
else:
    BATCH = 50
    for i in range(0, len(additions), BATCH):
        chunk = additions[i : i + BATCH]
        payload = {
            "feeds": [
                {"url": item["url"], "category_hint": item.get("category", "general")}
                for item in chunk
            ]
        }
        resp = httpx.post(
            f"{AGENT_API}/internal/feeds",
            json=payload,
            headers={"x-internal-token": INTERNAL_API_TOKEN},
            timeout=15.0,
        )
        resp.raise_for_status()
        data = resp.json()
        total_queued += data.get("queued", 0)
        total_skipped += data.get("skipped", 0)
        all_errors += data.get("errors", [])

    print(f"{GREEN}Additions applied{RESET}")
    print(f"  Queued  : {total_queued}")
    print(f"  Skipped : {total_skipped}  (already registered or duplicate)")
    if all_errors:
        print("  Errors  :")
        for e in all_errors:
            print(f"    {e}")

HTTPStatusError: Client error '401 Unauthorized' for url 'https://agent.serendip.mountainman.tech/internal/feeds'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [ ]:
# ── Cell 6: Apply removals -- mark feeds dead in rss_feeds ───────────────────
import asyncio
import hashlib

try:
    import psycopg
    from psycopg.rows import dict_row
except ModuleNotFoundError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psycopg[binary]"])
    import psycopg
    from psycopg.rows import dict_row


async def _mark_dead(urls):
    if not DATABASE_URL:
        raise RuntimeError("DATABASE_URL is not set -- update Cell 1.")
    conn = await psycopg.AsyncConnection.connect(DATABASE_URL, row_factory=dict_row)
    marked = not_found = 0
    async with conn:
        for url in urls:
            h = hashlib.sha256(url.encode()).hexdigest()
            async with conn.cursor() as cur:
                await cur.execute(
                    "UPDATE rss_feeds SET status = 'dead' WHERE url_hash = %s",
                    (h,),
                )
                if cur.rowcount > 0:
                    marked += 1
                else:
                    not_found += 1
        await conn.commit()
    return {"marked_dead": marked, "not_found": not_found}


if not removals:
    print("No removals to apply.")
else:
    result = asyncio.run(_mark_dead([item["url"] for item in removals]))
    print(f"{GREEN}Removals applied{RESET}")
    print(f"  Marked dead : {result['marked_dead']}")
    if result["not_found"]:
        print(f"  {YELLOW}Not in rss_feeds (never registered): {result['not_found']}{RESET}")

No removals to apply.


In [43]:
# ── Cell 7: Summary ──────────────────────────────────────────────────────────
from datetime import datetime

print(f"{BOLD}Feed patch applied -- {datetime.now().strftime('%Y-%m-%d %H:%M')}{RESET}")
print(f"  Source    : {PATCH_FILE.name}")
print(f"  Additions : {total_queued} queued, {total_skipped} skipped")
print(f"  Removals  : {len(removals)} feeds marked dead")
print(f"  Watch     : {len(watch)} (no action -- review manually)")
print()
print("Newly registered feeds will be harvested on the next refresh_seeds run (hourly).")

Feed patch applied -- 2026-04-19 10:40
  Source    : 2026-04-19-additions.yaml
  Additions : 14 queued, 0 skipped
  Removals  : 0 feeds marked dead
  Watch     : 3 (no action -- review manually)

Newly registered feeds will be harvested on the next refresh_seeds run (hourly).


In [ ]:
# ── Cell 8: Export indexed sites from rss_feeds ──────────────────────────────
# Dumps all active (or all) feeds to a YAML file you can use as a backup or
# seed list for another environment.
#
# Set EXPORT_STATUS = None to export every feed regardless of status.

import asyncio
import csv
from datetime import datetime
from pathlib import Path

EXPORT_STATUS = "active"  # "active" | "paused" | "dead" | None (all)
EXPORT_FORMAT = "yaml"  # "yaml" | "csv"
EXPORT_DIR = Path("~/Documents/Claude/Projects/Stumble Upon/rss-curator/exports").expanduser()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%dT%H%M")
label = EXPORT_STATUS or "all"
export_path = EXPORT_DIR / f"{timestamp}-feeds-{label}.{EXPORT_FORMAT}"


async def _export_feeds(status_filter):
    if not DATABASE_URL:
        raise RuntimeError("DATABASE_URL is not set -- update Cell 1.")
    import psycopg
    from psycopg.rows import dict_row

    conn = await psycopg.AsyncConnection.connect(DATABASE_URL, row_factory=dict_row)
    async with conn:
        async with conn.cursor() as cur:
            if status_filter:
                await cur.execute(
                    "SELECT url, category_hint, status, added_at, last_harvested_at, last_item_count "
                    "FROM rss_feeds WHERE status = %s ORDER BY category_hint, url",
                    (status_filter,),
                )
            else:
                await cur.execute(
                    "SELECT url, category_hint, status, added_at, last_harvested_at, last_item_count "
                    "FROM rss_feeds ORDER BY category_hint, url"
                )
            return await cur.fetchall()


rows = asyncio.run(_export_feeds(EXPORT_STATUS))

if EXPORT_FORMAT == "yaml":
    try:
        import yaml
    except ModuleNotFoundError:
        import subprocess
        import sys

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
        import yaml

    data = {
        "exported_at": timestamp,
        "status_filter": EXPORT_STATUS or "all",
        "count": len(rows),
        "feeds": [
            {
                "url": r["url"],
                "category": r["category_hint"],
                "status": r["status"],
                "added_at": r["added_at"].isoformat() if r["added_at"] else None,
                "last_harvested_at": r["last_harvested_at"].isoformat()
                if r["last_harvested_at"]
                else None,
                "last_item_count": r["last_item_count"],
            }
            for r in rows
        ],
    }
    export_path.write_text(yaml.dump(data, allow_unicode=True, sort_keys=False))

else:  # csv
    with export_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "url",
                "category_hint",
                "status",
                "added_at",
                "last_harvested_at",
                "last_item_count",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

print(f"{GREEN}Exported {len(rows)} feeds ({label}){RESET}")
print(f"  File: {export_path}")